# Cropped QH spectrum: periodic \(x\), finite \(y\), \(E_n\) vs \(\langle y\rangle_n\)

This notebook uses the same sweep-folder loading, source/drain reshaping, x-cropping, and two-stage \(U/\mu\) energy-reference convention as the conductance workflow.

The intended spectral geometry is
\[
\boxed{x\ \mathrm{periodic},\qquad y\ \mathrm{finite/open}.}
\]
Accordingly, every spectrum plot uses the ordinary transverse expectation value
\[
\langle y\rangle_n=\langle\psi_n|\hat y|\psi_n\rangle.
\]
Because \(y\) is not periodic, no circular mean is used.

For the magnetic field we use Landau gauge
\[
\mathbf A=(-By,0,0),
\]
so the Peierls phase is carried by the x-hopping and depends only on y. At fixed y the phase is constant along the whole periodic x direction, including the seam connecting the last x-column back to the first.


In [ ]:
from pathlib import Path
import copy
import dataclasses
import multiprocessing

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

import scipy.sparse.linalg as spla
import scipy.linalg as la

from joblib import Parallel, delayed

from qh_kwant_transport_mu_metadata_two_terminal_v4 import *
from qh_periodic_x_spectrum import (
    DEFAULT_PEIERLS_SIGN,
    build_periodic_x_hamiltonian,
    mean_y_from_probabilities,
    variance_y_from_probabilities,
    hermiticity_error,
    periodic_x_potential_mismatch,
    plaquette_phase,
    expected_plaquette_phase,
    benchmark_zero_field_periodic_x,
)

PEIERLS_SIGN = DEFAULT_PEIERLS_SIGN


In [ ]:
def cyclotron_energy_meV(B_T, m_eff=0.067):

    hbar = 1.054571817e-34  # J s

    e = 1.602176634e-19    # C

    m_e = 9.1093837015e-31 # kg

    hw_J = hbar * e * B_T / (m_eff * m_e)

    return hw_J / e * 1e3

## 1. Load the \(B\) sweep and set \(\min U=0\) frame by frame


In [ ]:
SWEEP_DIR = Path("../sweepone/data_sweep")
POTENTIAL_KEY = "U"

seq_raw = load_effective_potential_sequence(
    SWEEP_DIR,
    potential_key=POTENTIAL_KEY,
    ground_each_frame=False,
)

U_min_raw_allB = np.nanmin(seq_raw.U_stack_meV, axis=(1, 2))

U_stack_grounded = (
    np.asarray(seq_raw.U_stack_meV, dtype=float)
    - U_min_raw_allB[:, None, None]
)

seq = dataclasses.replace(
    seq_raw,
    U_stack_meV=U_stack_grounded,
)

selector = hartree_energy_selector(
    key_candidates=[
        "params.mu",
        "mu",
        "chemical_potential",
        "chemical_potential_meV",
        "fermi_energy",
        "fermi_energy_meV",
    ],
    unit="meV",
)

print(seq.U_stack_meV.shape, "(nB, ny, nx)")
print("B range:", seq.B_values_T[0], "->", seq.B_values_T[-1], "T")
print("x range:", seq.x_nm.min(), "->", seq.x_nm.max(), "nm")
print("y range:", seq.y_nm.min(), "->", seq.y_nm.max(), "nm")
print("a =", seq.a_nm, "nm")
print(
    "largest residual |min(U)| after grounding =",
    np.max(np.abs(np.nanmin(seq.U_stack_meV, axis=(1, 2)))),
    "meV",
)


## 2. Prepare and crop the potential exactly as in the transport workflow


In [ ]:
source_drain_depth_nm = 800.0

lead_potential_meV = 0.0
smooth_nm = 1e1

lead_specs_2 = (
    LeadSpec(
        "source", "left",
        depth_nm=source_drain_depth_nm,
        lead_potential_meV=lead_potential_meV,
        smooth_nm=smooth_nm,
        smooth_profile="constant_inner",
    ),
    LeadSpec(
        "drain", "right",
        depth_nm=source_drain_depth_nm,
        lead_potential_meV=lead_potential_meV,
        smooth_nm=smooth_nm,
        smooth_profile="constant_inner",
    ),
)

cfg2 = DeviceConfig(
    m_eff=0.067,
    g_factor=-0.0,
    norbs=1,
    disorder=DisorderConfig(
        amplitude_meV=1e-6,
        correlation_nm=30.0,
        seed=12345,
    ),
    lead_specs=lead_specs_2,
    reshape_under_leads=True,
    lead_B_mode="same",
)

_prepare_frame_full = prepare_frame


def crop_prepared_frame_x(prep, crop_depth_nm):
    """
    Return a copy of a v4 PreparedFrame restricted to the inner x interval

        xmin + crop_depth_nm < x < xmax - crop_depth_nm.

    The crop is applied to x_nm and to every prepared ndarray whose final
    spatial dimensions are (ny, nx), plus any 1D x-dependent array.

    This function deliberately acts AFTER the original v4 prepare_frame,
    hence U_for_transport_meV has already undergone the original lead-mouth
    reshaping before the outer regions are thrown away.
    """
    import copy
    import dataclasses

    x = np.asarray(prep.x_nm)
    y = np.asarray(prep.y_nm)
    nx = len(x)
    ny = len(y)

    xmin = float(x.min())
    xmax = float(x.max())

    x_left = xmin + float(crop_depth_nm)
    x_right = xmax - float(crop_depth_nm)

    keep_x = (x > x_left) & (x < x_right)

    if np.count_nonzero(keep_x) < 2:
        raise ValueError(
            "source_drain_depth_nm removes the whole transport region: "
            f"full x range = [{xmin}, {xmax}] nm, "
            f"requested retained interval = ({x_left}, {x_right}) nm."
        )

    def crop_value(name, value):
        # Coordinate arrays must be treated explicitly.
        if name == "x_nm":
            return np.asarray(value)[keep_x]

        if name == "y_nm":
            return np.asarray(value)   # NEVER crop y

        if not isinstance(value, np.ndarray):
            return value

        # Spatial maps: crop ONLY their x axis.
        if value.ndim >= 2 and value.shape[-2:] == (ny, nx):
            sl = [slice(None)] * value.ndim
            sl[-1] = keep_x
            return value[tuple(sl)]

        # 1D arrays known to live on x.
        # Only use this generic rule after excluding y_nm explicitly.
        if value.ndim == 1 and value.shape[0] == nx:
            return value[keep_x]

        return value

    if dataclasses.is_dataclass(prep):
        updates = {}
        for f in dataclasses.fields(prep):
            old = getattr(prep, f.name)
            new = crop_value(f.name, old)
            if new is not old:
                updates[f.name] = new
        cropped = dataclasses.replace(prep, **updates)
    else:
        # Fallback for a non-dataclass PreparedFrame implementation.
        cropped = copy.copy(prep)
        for name, old in vars(prep).items():
            new = crop_value(name, old)
            if new is not old:
                setattr(cropped, name, new)

    return cropped


def cropped_boundary_reference(prep_cropped):
    x = np.asarray(prep_cropped.x_nm)
    y = np.asarray(prep_cropped.y_nm)
    U = np.asarray(prep_cropped.U_for_transport_meV)

    iy0 = int(np.argmin(np.abs(y)))
    U_left = float(U[iy0, 0])
    U_right = float(U[iy0, -1])
    U_boundary = 0.5 * (U_left + U_right)

    return U_boundary, U_left, U_right, float(x[0]), float(x[-1]), float(y[iy0])


def shift_transport_potential(prep_cropped, shift_meV):
    U_new = (
        np.asarray(prep_cropped.U_for_transport_meV, dtype=float)
        - float(shift_meV)
    )
    return dataclasses.replace(
        prep_cropped,
        U_for_transport_meV=U_new,
    )


def prepare_spectrum_frame(index):
    prep_full = _prepare_frame_full(seq, index, cfg2)
    prep_crop = crop_prepared_frame_x(
        prep_full,
        source_drain_depth_nm,
    )

    (
        U_boundary,
        U_left,
        U_right,
        x_left,
        x_right,
        y0,
    ) = cropped_boundary_reference(prep_crop)

    prep_final = shift_transport_potential(
        prep_crop,
        U_boundary,
    )

    mu_raw = float(selector(seq_raw.frame(index)))
    U_min_raw = float(U_min_raw_allB[index])
    mu_minref = mu_raw - U_min_raw
    mu_final = mu_minref - U_boundary

    refs = {
        "mu_raw_meV": mu_raw,
        "U_min_raw_meV": U_min_raw,
        "mu_minref_meV": mu_minref,
        "U_boundary_meV": U_boundary,
        "mu_final_meV": mu_final,
        "U_left_before_boundary_shift_meV": U_left,
        "U_right_before_boundary_shift_meV": U_right,
        "x_left_nm": x_left,
        "x_right_nm": x_right,
        "y_center_nm": y0,
    }

    return prep_final, refs


## 3. Boundary-condition and magnetic-phase benchmarks


In [ ]:
# 1) Exact benchmark of periodic-x / open-y connectivity at B=0, U=0.
bc_test = benchmark_zero_field_periodic_x(nx=8, ny=7, t=1.0)
print(
    "B=0 analytic periodic-x/open-y benchmark: "
    f"max |E_num-E_exact| = {bc_test['max_abs_error']:.3e}"
)

# 2) Benchmark a real prepared frame.
BENCHMARK_FRAME = min(250, len(seq.B_values_T) - 1)
prep_bench, refs_bench = prepare_spectrum_frame(BENCHMARK_FRAME)

H_bench, _, _ = build_periodic_x_hamiltonian(
    prep_bench,
    periodic_x=True,
    peierls_sign=PEIERLS_SIGN,
)

seam = periodic_x_potential_mismatch(prep_bench)
phase_num = plaquette_phase(prep_bench, iy=0, peierls_sign=PEIERLS_SIGN)
phase_expected = expected_plaquette_phase(prep_bench, peierls_sign=PEIERLS_SIGN)

print()
print("real-frame benchmark")
print("B =", prep_bench.B_T, "T")
print("alpha =", prep_bench.alpha)
print("Hamiltonian dimension =", H_bench.shape)
print("Hermiticity error =", hermiticity_error(H_bench), "meV")
print("plaquette phase =", phase_num, "rad")
print("expected phase  =", phase_expected, "rad")
print("phase error     =", np.angle(np.exp(1j*(phase_num-phase_expected))), "rad")
print()
print("periodic-x potential seam:")
print("max |U(xmax)-U(xmin)| =", seam["max_abs_meV"], "meV")
print("rms mismatch           =", seam["rms_meV"], "meV")
print("mean mismatch          =", seam["mean_meV"], "meV")


In [ ]:
# Visualize the two x-columns that are joined by the periodic seam.
plt.figure(figsize=(7, 4))
plt.plot(
    prep_bench.y_nm * 1e-3,
    prep_bench.U_for_transport_meV[:, 0],
    label=r"$U(x_{\min},y)$",
)
plt.plot(
    prep_bench.y_nm * 1e-3,
    prep_bench.U_for_transport_meV[:, -1],
    label=r"$U(x_{\max},y)$",
)
plt.xlabel(r"$y$ [$\mu$m]")
plt.ylabel("U [meV]")
plt.title("Potential on the two x-columns identified by periodic x")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 3.5))
plt.plot(
    prep_bench.y_nm * 1e-3,
    seam["difference_meV"],
)
plt.axhline(0.0, ls="--", lw=1)
plt.xlabel(r"$y$ [$\mu$m]")
plt.ylabel(r"$U(x_{\max},y)-U(x_{\min},y)$ [meV]")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
plt.pcolormesh(
    prep_bench.x_nm,
    prep_bench.y_nm,
    prep_bench.U_for_transport_meV,
    shading="auto",
    vmin=0,
    vmax=10,
)
plt.colorbar(label="meV")
plt.xlabel("x [nm]")
plt.ylabel("y [nm]")
plt.show()
print(np.amin(prep_bench.U_for_transport_meV))

## 4. Diagonalize one frame

By default the notebook uses sparse shift-invert diagonalization around the final shifted chemical potential. This is much cheaper than computing the complete spectrum of a large two-dimensional grid.


In [ ]:
#FRAME_INDEX = min(250, len(seq.B_values_T) - 1)
FRAME_INDEX = 90

DIAGONALIZATION_MODE = "near_mu"  # "near_mu" or "full"
N_EIGS = 1700
ENERGY_WINDOW_AROUND_MU_MEV = None


def diagonalize_frame(index):
    prep, refs = prepare_spectrum_frame(index)

    H, x_site_nm, y_site_nm = build_periodic_x_hamiltonian(
        prep,
        periodic_x=True,
        peierls_sign=PEIERLS_SIGN,
    )

    mu = (float(refs["mu_final_meV"])- np.amin(prep.U_for_transport_meV) )      *1
    n = H.shape[0]

    if DIAGONALIZATION_MODE == "full":
        evals, evecs = la.eigh(H.toarray())
    elif DIAGONALIZATION_MODE == "near_mu":
        if n <= 2:
            raise ValueError("System is too small to diagonalize.")
        k = min(int(N_EIGS), n - 2)
        evals, evecs = spla.eigsh(H, k=k, sigma=mu, which="LM")
        order = np.argsort(evals)
        evals = np.real(evals[order])
        evecs = evecs[:, order]
    else:
        raise ValueError("DIAGONALIZATION_MODE must be 'near_mu' or 'full'.")

    probabilities = np.abs(evecs) ** 2
    y_expect_nm = mean_y_from_probabilities(probabilities, y_site_nm)
    y_var_nm2 = variance_y_from_probabilities(probabilities, y_site_nm)

    if ENERGY_WINDOW_AROUND_MU_MEV is not None:
        w = float(ENERGY_WINDOW_AROUND_MU_MEV)
        keep = np.abs(evals - mu) <= w
        evals = evals[keep]
        y_expect_nm = y_expect_nm[keep]
        y_var_nm2 = y_var_nm2[keep]

    seam = periodic_x_potential_mismatch(prep)

    return {
        "index": int(index),
        "B_T": float(prep.B_T),
        "mu_meV": refs["mu_final_meV"],
        "eigenenergies_meV": np.asarray(evals, dtype=float),
        "y_expect_nm": np.asarray(y_expect_nm, dtype=float),
        "y_std_nm": np.sqrt(np.asarray(y_var_nm2, dtype=float)),
        "y_min_nm": float(np.min(prep.y_nm)),
        "y_max_nm": float(np.max(prep.y_nm)),
        "n_sites": int(H.shape[0]),
        "hermiticity_error_meV": hermiticity_error(H),
        "seam_max_abs_meV": seam["max_abs_meV"],
        "seam_rms_meV": seam["rms_meV"],
        "refs": refs,
    }


test = diagonalize_frame(FRAME_INDEX)

print("B =", test["B_T"], "T")
print("number of sites =", test["n_sites"])
print("number of eigenvalues returned =", len(test["eigenenergies_meV"]))
print("mu =", test["mu_meV"], "meV")
print("Hermiticity error =", test["hermiticity_error_meV"], "meV")
print("periodic-x seam max mismatch =", test["seam_max_abs_meV"], "meV")




In [ ]:

hwB_meV = cyclotron_energy_meV(
    test["B_T"],
    m_eff=cfg2.m_eff,
)
plt.figure(figsize=(7, 5))
plt.scatter(
    test["y_expect_nm"] * 1e-3,
    test["eigenenergies_meV"],
    s=12,
)
plt.axhline(
    test["mu_meV"],
    ls="--",
    label=rf"$\mu={test['mu_meV']:.3f}$ meV",
)
plt.xlabel(r"$\langle y\rangle$ [$\mu$m]")
plt.ylabel("eigenenergy [meV]")
plt.title(rf"$B={test['B_T']:.4f}$ T, periodic $x$")
plt.grid(True)
plt.legend()
plt.ylim(0,20)
plt.tight_layout()
plt.show()
print(hwB_meV)

## 5. Diagonalize the selected \(B\) sweep


In [ ]:
B_STRIDE = 10
indices = list(range(0, len(seq.B_values_T), B_STRIDE))

N_JOBS = max(1, min(4, multiprocessing.cpu_count() - 1))

spectrum_results = Parallel(
    n_jobs=N_JOBS,
    backend="loky",
    verbose=10,
)(
    delayed(diagonalize_frame)(i)
    for i in indices
)

spectrum_results = sorted(spectrum_results, key=lambda r: r["index"])
B_values = np.array([r["B_T"] for r in spectrum_results])
mu_values = np.array([r["mu_meV"] for r in spectrum_results])

print("finished", len(spectrum_results), "B frames")
print(
    "states/frame:",
    min(len(r["eigenenergies_meV"]) for r in spectrum_results),
    "->",
    max(len(r["eigenenergies_meV"]) for r in spectrum_results),
)
print(
    "largest periodic-x potential mismatch over sweep =",
    max(r["seam_max_abs_meV"] for r in spectrum_results),
    "meV",
)


## 6. Animate \(E_n\) versus \(\langle x
angle_n\)


In [ ]:
VIDEO_FILE = "cropped_periodic_x_spectrum_vs_y.mp4"
VIDEO_FPS = 8
VIDEO_DPI = 150

ymin_nm = min(r["y_min_nm"] for r in spectrum_results)
ymax_nm = max(r["y_max_nm"] for r in spectrum_results)

all_E = np.concatenate([r["eigenenergies_meV"] for r in spectrum_results])
all_mu = np.asarray(mu_values)
Emin = float(min(np.min(all_E), np.min(all_mu)))
Emax = float(max(np.max(all_E), np.max(all_mu)))
pad = 0.05 * max(Emax - Emin, 1e-9)
Emin -= pad
Emax += pad

fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter([], [], s=12)
mu_line = ax.axhline(0.0, ls="--", lw=1.5)

ax.set_xlim(ymin_nm * 1e-3, ymax_nm * 1e-3)
ax.set_ylim(Emin, Emax)
ax.set_xlabel(r"$\langle y\rangle$ [$\mu$m]")
ax.set_ylabel("eigenenergy [meV]")
ax.grid(True)
title = ax.set_title("")


def update(frame_number):
    r = spectrum_results[frame_number]
    xy = np.column_stack((r["y_expect_nm"] * 1e-3, r["eigenenergies_meV"]))
    scatter.set_offsets(xy)
    mu = float(r["mu_meV"])
    mu_line.set_ydata([mu, mu])
    title.set_text(
        rf"$B={r['B_T']:.4f}$ T   "
        rf"$\mu={mu:.4f}$ meV   "
        rf"$N={len(r['eigenenergies_meV'])}$"
    )
    return scatter, mu_line, title


anim = FuncAnimation(
    fig,
    update,
    frames=len(spectrum_results),
    interval=1000 / VIDEO_FPS,
    blit=False,
)

try:
    anim.save(VIDEO_FILE, writer="ffmpeg", fps=VIDEO_FPS, dpi=VIDEO_DPI)
    print("saved", VIDEO_FILE)
except Exception as exc:
    GIF_FILE = Path(VIDEO_FILE).with_suffix(".gif")
    print("ffmpeg save failed:", exc)
    print("trying Pillow GIF instead:", GIF_FILE)
    anim.save(GIF_FILE, writer="pillow", fps=VIDEO_FPS, dpi=VIDEO_DPI)
    print("saved", GIF_FILE)

plt.close(fig)


## 7. Save spectrum and benchmark diagnostics


In [ ]:
SPECTRUM_DATA_FILE = "cropped_periodic_x_spectrum_data.npz"

np.savez(
    SPECTRUM_DATA_FILE,
    indices=np.array([r["index"] for r in spectrum_results], dtype=int),
    B=B_values,
    mu=mu_values,
    eigenenergies=np.array([r["eigenenergies_meV"] for r in spectrum_results], dtype=object),
    y_expect_nm=np.array([r["y_expect_nm"] for r in spectrum_results], dtype=object),
    y_std_nm=np.array([r["y_std_nm"] for r in spectrum_results], dtype=object),
    n_sites=np.array([r["n_sites"] for r in spectrum_results], dtype=int),
    hermiticity_error_meV=np.array([r["hermiticity_error_meV"] for r in spectrum_results], dtype=float),
    seam_max_abs_meV=np.array([r["seam_max_abs_meV"] for r in spectrum_results], dtype=float),
    seam_rms_meV=np.array([r["seam_rms_meV"] for r in spectrum_results], dtype=float),
    source_drain_depth_nm=float(source_drain_depth_nm),
    peierls_sign=float(PEIERLS_SIGN),
    zero_field_bc_benchmark_error=float(bc_test["max_abs_error"]),
)
print("saved", SPECTRUM_DATA_FILE)
